# Delta table Structure

## Delta Tables
- Storage is in Parquet files
- Delta lake service uses a transaction log
- Any insert, update, or delete will create a new file

In [0]:
# Configure database
spark.sql('CREATE DATABASE IF NOT EXISTS workspace.stage_wwi')
spark.sql('USE workspace.stage_wwi')
spark.sql('DROP TABLE IF EXISTS invoice')
spark.sql('CREATE VOLUME IF NOT EXISTS workspace.stage_wwi.raw_data')

In [0]:
spark.sql('USE workspace.stage_wwi')

# File parameters
file_path = '/Volumes/workspace/stage_wwi/raw_data/Invoice.csv'
file_format = 'csv'
infer_schema = 'true'
header = 'true'
delimiter = ','

# Reading CSV file
dfInvoice = spark.read \
    .format(file_format) \
    .option('inferSchema', infer_schema) \
    .option('header', header) \
    .option('delimiter', delimiter) \
    .load(file_path)

dfInvoice.limit(10).display()

In [0]:
from pyspark.sql import functions as sf

def trunc_date(date_column, date_trunc, output_format):
    output = sf.date_format(
        sf.date_trunc(date_trunc, date_column),
        output_format
    )
    return output

# Truncate date to year
dfInvoice = dfInvoice.withColumn('InvoiceYear', trunc_date(dfInvoice.InvoiceDate, 'year', 'yyyy'))

# Truncate date to quarter
dfInvoice = dfInvoice.withColumn('InvoiceQuarter', trunc_date(dfInvoice.InvoiceDate, 'quarter', 'yyyyQQ'))

# Truncate date to month
dfInvoice = dfInvoice.withColumn('InvoiceMonth', trunc_date(dfInvoice.InvoiceDate, 'month', 'yyyyMM'))


dfInvoice.limit(10).display()

In [0]:
dfInvoice[['InvoiceDate', 'InvoiceMonth', 'InvoiceQuarter', 'InvoiceYear']].distinct().display()

In [0]:
# Creating subset of data for 2021
df2021 = dfInvoice.filter(dfInvoice.InvoiceYear == '2021')

# Loading data into delta table
df2021.write.format('delta').mode('overwrite').saveAsTable('invoice')

In [0]:
# The DBFS root is disabled in this workspace, so %fs cannot be used.
# DESCRIBE DETAIL shows the Delta table's file-level info (format, numFiles, sizeInBytes).
display(spark.sql('DESCRIBE DETAIL workspace.stage_wwi.invoice'))

In [0]:
# Creating subset of data for 2022
df2022 = dfInvoice.filter(dfInvoice.InvoiceYear == '2022')

# Loading data into delta table
df2022.write.format('delta').mode('append').saveAsTable('invoice')

In [0]:
%sql
-- The DBFS root is disabled in this workspace, so %fs cannot be used.
-- DESCRIBE DETAIL shows the Delta table's file-level info (format, numFiles, sizeInBytes).
DESCRIBE DETAIL workspace.stage_wwi.invoice

In [0]:
%sql
DESCRIBE HISTORY workspace.stage_wwi.invoice

In [0]:
# Creating subset of data for 2023
df2023 = dfInvoice.filter(dfInvoice.InvoiceYear == '2023')

# Loading data into delta table
df2023.write.format('delta').mode('append').saveAsTable('invoice')

In [0]:
%sql
-- The DBFS root is disabled in this workspace, so %fs cannot be used.
-- DESCRIBE DETAIL shows the Delta table's file-level info (format, numFiles, sizeInBytes).
DESCRIBE DETAIL workspace.stage_wwi.invoice

In [0]:
%sql
DESCRIBE HISTORY workspace.stage_wwi.invoice

In [0]:
# Creating subset of data for 2024 Q1
df2024Q1 = dfInvoice.filter(dfInvoice.InvoiceQuarter == '202401')

# Loading data into delta table
df2024Q1.write.format('delta').mode('append').saveAsTable('invoice')

In [0]:
%sql
DESCRIBE HISTORY workspace.stage_wwi.invoice

In [0]:
# Creating subset of data for 2024 April
df202404 = dfInvoice.filter(dfInvoice.InvoiceMonth == '202404')

# Loading data into delta table
df202404.write.format('delta').mode('append').saveAsTable('invoice')

In [0]:
%sql
DESCRIBE HISTORY workspace.stage_wwi.invoice

## Conclusion
The more frequent and granular the data loads are, the larger number o files with small amount of rows. After some time, multiple small files can impact the system performance, so a file optmization may be required

Before Optimization - 5 files

In [0]:
%sql
-- The DBFS root is disabled in this workspace, so %fs cannot be used.
-- DESCRIBE DETAIL shows the Delta table's file-level info (format, numFiles, sizeInBytes).
DESCRIBE DETAIL workspace.stage_wwi.invoice

Optimizing Delta table

In [0]:
%sql
OPTIMIZE workspace.stage_wwi.invoice

After Optimization - 1 file

In [0]:
%sql
-- The DBFS root is disabled in this workspace, so %fs cannot be used.
-- DESCRIBE DETAIL shows the Delta table's file-level info (format, numFiles, sizeInBytes).
DESCRIBE DETAIL workspace.stage_wwi.invoice